<a href="https://colab.research.google.com/github/adam1brownell/Chat-Program/blob/master/LLMatchmaker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import requests
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from getpass import getpass

os.environ['HF_TOKEN'] = getpass('Enter your HF API key: ')

Enter your HF API key: ··········


In [14]:
%%capture
!pip install langchain
!pip install langchain_community
!pip install langchain_openai
!pip install langchain_huggingface
!pip install langchain-chroma

# # Installs Unsloth, Xformers (Flash Attention) and all other packages!
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
# !pip install triton

In [15]:
from transformers import BertModel, BertTokenizer
import torch
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings
# OPENAI_KEY = getpass('Enter Open AI API Key: ')
# os.environ['OPENAI_API_KEY'] = OPENAI_KEY

In [3]:
## Restart Runtime

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
okc_pd = pd.read_csv('/content/drive/MyDrive/LLMatchmaker/okcupid.csv')

okc_pd["p_id"]=okc_pd.index

cols = ["p_id"]
essay_cols = ["essay"+str(i) for i in range(10)]
okc_text = okc_pd[cols+essay_cols]

"""
essay0- My self summary
essay1- What I’m doing with my life
essay2- I’m really good at
essay3- The first thing people usually notice about me
essay4- Favorite books, movies, show, music, and food
essay5- The six things I could never do without
essay6- I spend a lot of time thinking about
essay7- On a typical Friday night I am
essay8- The most private thing I am willing to admit
essay9- You should message me if...
"""

okc_text['document_str'] = okc_text[essay_cols].apply(lambda x: '\n'.join(x.astype(str)), axis=1)
okc_doc = okc_text[['p_id', 'document_str']]

<ipython-input-7-98ace10c877e>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  okc_text['document_str'] = okc_text[essay_cols].apply(lambda x: '\n'.join(x.astype(str)), axis=1)


In [8]:
# from unsloth import FastLanguageModel
# import torch
# max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
# dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
# load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# model, tokenizer = FastLanguageModel.from_pretrained(model_name = "unsloth/llama-3-8b-bnb-4bit", # YOUR MODEL YOU USED FOR TRAINING
#         max_seq_length = max_seq_length,
#         dtype = dtype,
#         load_in_4bit = load_in_4bit,
# )
# FastLanguageModel.for_inference(model)

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
# from peft import LoraConfig, get_peft_model
from huggingface_hub import login


# Login to Hugging Face Hub
login(token= os.environ['HF_TOKEN'])

# Load the pre-trained LLaMA 3 model and tokenizer
model_name = "meta-llama/Meta-Llama-3-8B"
# model_name = "EleutherAI/gpt-neo-1.3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({'eos_token': 'Human:'})
model = AutoModelForCausalLM.from_pretrained(model_name)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

In [10]:
 okc_doc['document_vect'] = okc_doc['document_str'].apply(lambda x: tokenizer(x, return_tensors="pt")['input_ids'])

<ipython-input-10-8f7467505b62>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  okc_doc['document_vect'] = okc_doc['document_str'].apply(lambda x: tokenizer(x, return_tensors="pt")['input_ids'])


In [16]:
okc_doc.head()

,p_id,document_str,document_vect
0,0,about me: i would love to think that i was so...,"[[tensor(128000), tensor(9274), tensor(757), t..."
1,1,i am a chef: this is what that means. 1. i am ...,"[[tensor(128000), tensor(72), tensor(1097), te..."
2,2,"i'm not ashamed of much, but writing public te...","[[tensor(128000), tensor(72), tensor(2846), te..."
3,3,i work in a library and go to school. . .\nrea...,"[[tensor(128000), tensor(72), tensor(990), ten..."
4,4,hey how's it going? currently vague on the pro...,"[[tensor(128000), tensor(36661), tensor(1268),..."


In [24]:
# import torch

## GEMINI FAIL

# def cosine_sim(a, b):
#   return torch.nn.functional.cosine_similarity(a, b, dim=1)
# def euclidean_dist(a, b):
#   return torch.cdist(a, b, p=2)
# def manhattan_dist(a, b):
#   return torch.cdist(a, b, p=1)

# def get_closest_documents(inputs, document_embeddings, sim="cosine"):
#   if sim == "cosine":
#     similarities = cosine_sim(inputs, document_embeddings)
#   elif sim == "euclidean":
#     similarities = euclidean_dist(inputs, document_embeddings)
#   elif sim == "manhattan":
#     similarities = manhattan_dist(inputs, document_embeddings)
#   else:
#     raise ValueError("Invalid similarity metric")
#   closest_indices = torch.argsort(similarities, descending=True)
#   return closest_indices

# # Example usage
# input_text = "I want to date someone who is funny and honest"
# inputs = tokenizer(input_text, return_tensors="pt")
# print(inputs)

# document_tensors = torch.cat(list(okc_doc.document_vect), dim=0)
# closest_docs = get_closest_documents(inputs['input_ids'], document_tensors)

# # You can access the indices of the closest documents like this:
# print(closest_docs)

In [33]:
# Sorta following this RAG Guide: https://www.analyticsvidhya.com/blog/2024/07/building-agentic-rag-systems-with-langgraph/

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
# openai_embed_model = OpenAIEmbeddings(model='text-embedding-ada-002')


# Create docs
docs = [Document(page_content=row['document_str'],
                 metadata={'p_id': row["p_id"]}) for _, row in okc_doc.iterrows()]

# Chunk docs
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=300)
chunked_docs = splitter.split_documents(docs)
chunked_docs[:3]

[Document(metadata={'p_id': 0}, page_content="about me:  i would love to think that i was some some kind of intellectual: either the dumbest smart guy, or the smartest dumb guy. can't say i can tell the difference. i love to talk about ideas and concepts. i forge odd metaphors instead of reciting cliches. like the simularities between a friend of mine's house and an underwater salt mine. my favorite word is salt by the way (weird choice i know). to me most things in life are better as metaphors. i seek to make myself a little better everyday, in some productively lazy way. got tired of tying my shoes. considered hiring a five year old, but would probably have to tie both of our shoes... decided to only wear leather shoes dress shoes.  about you:  you love to have really serious, really deep conversations about really silly stuff. you have to be willing to snap me out of a light hearted rant with a kiss. you don't have to be funny, but you have to be able to make me laugh. you should be

In [32]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings()

vector_store = Chroma(
    collection_name="dating_profile_db",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
    persist_directory="./dating_profile_db"
)

In [35]:
def batch_iterable(iterable, batch_size):
    """Yield successive batches from the iterable."""
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i + batch_size]

# Example usage
batch_size = 2000
i = 0
for doc_batch in batch_iterable(chunked_docs, batch_size):
  if i % 10 == 0:
    print(i)
  try:
    vector_store.add_documents(doc_batch)
    i+=1
  except:
    break
print(f"processed {i*batch_size:,} document chunks")

0
10
20
30
40
50
processed 102,000 document chunks


In [36]:
query = "I want to date someone who is smart, funny, and honest. I like to hike and scream at the moon."
results = vector_store.similarity_search_with_score(
    query, k=3
)
for res, score in results:
    print(f"* [Profile #{res.metadata['p_id']}][SIM={score:2f}] {res.page_content}")
    print()

# retriever = vector_store.as_retriever(search_type="similarity_score_threshold",
#                        search_kwargs={"k": 3,
#                        "score_threshold": 0.99})
# retriever.invoke(query)

* [Profile #24982][SIM=0.278789] i'm a fun loving, energetic, fit and down-to-earth guy. i'm straight forward, honest, loyal and financially secure. i like to exercise, read, watch movies, paint and travel. i enjoy driving trips, live theatre, good food and fine wine. i laugh easily and have been told i have a great smile. i'm a good conversationalist, well read and east coast educated.
living day to day and enjoying every minute
finding fun things to do, outdoor activities and making money
my smile and light blue eyes
seabiscuit, unbroken, john adams, the new yorker, wall street journal american conservatory theatre rolling stones, p!nk, beatles, u2, lady gaga, van morrison italian, sushi, indian, barbecue
my daughter running biking reading windsurfing snowboarding traveling physical intimacy (ok, eight, but who's counting)
huh, what? why think when you can just do it
dating
i'm really quite shy despite my outgoing attitude
you like what you've read and see!

* [Profile #44073][SIM=0.

In [ ]:
from langchain import HuggingFaceEmbeddings, LangGraph, Chain

# HuggingFace LLM
huggingface_llm = HuggingFaceEmbeddings(model="gpt-neo-125M")

# List of additional prompts that might be used in step 3
prompt_list = [
    "What hobbies do you enjoy?",
    "What's the most important trait you value in a relationship?",
    "Tell me about your past relationships.",
    "What are your long-term goals?",
    "How do you handle conflict in relationships?"
]

# Dummy function to decide if more information is needed (step 4)
def has_enough_information(user_data):
    # This is a basic check, adjust it as per your logic
    return len(user_data) > 100  # Arbitrary threshold

# Step 1: Ask the user to tell about themselves
def ask_about_user():
    return huggingface_llm.embed_query("Tell me about yourself!")

# Step 2: Ask what they are looking for in a romantic partner
def ask_romantic_preference():
    return huggingface_llm.embed_query("Thanks for sharing. What are you looking for in a romantic partner?")

# Step 3: Choose an appropriate prompt based on user input
def choose_prompt(user_data):
    # Logic to decide the most relevant prompt from the list
    # For simplicity, we're randomly picking here, but you can use embeddings
    return huggingface_llm.embed_query(prompt_list[0])  # Example: choose the first prompt

# Step 4: Check if we have enough data, otherwise ask follow-up
def check_sufficient_data(user_data):
    if has_enough_information(user_data):
        return "We have enough data for a profile."
    else:
        return "I need a bit more information. Can you elaborate on your goals in life?"

# Step 5: Retrieve additional information (final step)
def get_rag():
    # Simulate a retrieval action, e.g., summarizing data or pulling external info
    return "Here is the final summary of your dating profile."

# Create the chain using LangGraph
chain = Chain()

# Add nodes to the chain
chain.add_node(ask_about_user, "Ask about the user")
chain.add_node(ask_romantic_preference, "Ask about romantic preferences")
chain.add_node(choose_prompt, "Choose appropriate prompt")
chain.add_node(check_sufficient_data, "Check if enough data is gathered")
chain.add_node(get_rag, "Retrieve final profile information")

# Define the graph connections (control flow)
# Assume each node returns a response and passes it to the next one
graph = LangGraph()

# Chain the steps together
graph.connect(ask_about_user, ask_romantic_preference)
graph.connect(ask_romantic_preference, choose_prompt)
graph.connect(choose_prompt, check_sufficient_data)
graph.connect(check_sufficient_data, get_rag)

# Run the chain
result = graph.run()
print(result)

In [29]:
# prompt: Use LangGraph to create a chain where the HuggingFace LLM "model" (1) asks the user to "tell me about yourself", then (2) asks "what are you looking for in a romantic partner?", then (3) decides on which prompt from a list "prompt_list" is most appropriate and asks that question. Then (4) determines if we have enough information on the user for a fully fleshed out dating profile, or not

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from transformers import pipeline

# Assuming 'model' is your HuggingFace LLM and 'tokenizer' is defined

# Create a pipeline for text generation
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
llm = HuggingFacePipeline(pipeline=pipe)

# Define prompts for each step
prompt_templates = [
    "Tell me about yourself.",
    "What are you looking for in a romantic partner?",
    # Add more prompts as needed
]

# Create a list of LLMChains, one for each prompt
chains = [
    LLMChain(llm=llm, prompt=PromptTemplate(template=template, input_variables=[]))
    for template in prompt_templates
]

# Define a function to determine the most appropriate prompt
def choose_prompt(user_responses):
    # Implement logic to select the best prompt based on user responses
    # This could involve analyzing the user's answers and choosing a prompt
    # that gathers more relevant information
    # For example, you could use a simple rule-based system or a more
    # complex machine learning model.
    return "prompt_list[0]"  # Replace with your logic

# Define a function to determine if we have enough information
def enough_information(user_responses):
    # Implement logic to determine if we have enough information
    # This could involve checking if the user has answered all the
    # necessary questions or if their answers are sufficiently detailed
    # For example, you could use a simple rule-based system or a more
    # complex machine learning model.
    return True  # Replace with your logic

# Run the chain
user_responses = []
for chain in chains:
    response = chain.run({})
    user_responses.append(response)

# Choose the next prompt
prompt = choose_prompt(user_responses)

# Ask the next question
response = llm(prompt)
user_responses.append(response)

# Check if we have enough information
if enough_information(user_responses):
    # Create a dating profile
    print("Enough information to create a dating profile")
else:
    # Ask more questions
    print("Need more information")



NameError: name 'chroma_db' is not defined